In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/movie-recommendation-hackathon-2026/sample_submission.csv
/kaggle/input/movie-recommendation-hackathon-2026/movies.csv
/kaggle/input/movie-recommendation-hackathon-2026/imdb_data.csv
/kaggle/input/movie-recommendation-hackathon-2026/genome_tags.csv
/kaggle/input/movie-recommendation-hackathon-2026/genome_scores.csv
/kaggle/input/movie-recommendation-hackathon-2026/train.csv
/kaggle/input/movie-recommendation-hackathon-2026/test.csv
/kaggle/input/movie-recommendation-hackathon-2026/tags.csv
/kaggle/input/movie-recommendation-hackathon-2026/links.csv


In [2]:
train = pd.read_csv(
    "/kaggle/input/movie-recommendation-hackathon-2026/train.csv",
    dtype={
        "userId": "int32",
        "movieId": "int32",
        "rating": "float32",
        "timestamp": "int64"
    }
)

test = pd.read_csv(
    "/kaggle/input/movie-recommendation-hackathon-2026/test.csv",
    dtype={
        "userId": "int32",
        "movieId": "int32"
    }
)

print("Train shape:", train.shape)
print("Test shape:", test.shape)


Train shape: (10000038, 4)
Test shape: (5000019, 2)


In [3]:
n_users = train["userId"].nunique()
n_movies = train["movieId"].nunique()
n_ratings = len(train)

sparsity = 1 - (n_ratings / (n_users * n_movies))

print("Number of users:", n_users)
print("Number of movies:", n_movies)
print("Number of ratings:", n_ratings)
print("Sparsity:", round(sparsity, 5))

print("\nMean rating:", round(train["rating"].mean(), 4))
print("Std rating:", round(train["rating"].std(), 4))


Number of users: 162541
Number of movies: 48213
Number of ratings: 10000038
Sparsity: 0.99872

Mean rating: 3.5334
Std rating: 1.0368


In [4]:
user_counts = train.groupby("userId")["rating"].count()
movie_counts = train.groupby("movieId")["rating"].count()

print("Average ratings per user:", round(user_counts.mean(), 2))
print("Average ratings per movie:", round(movie_counts.mean(), 2))

print("\nMost active user rated:", user_counts.max(), "movies")
print("Most rated movie has:", movie_counts.max(), "ratings")


Average ratings per user: 61.52
Average ratings per movie: 207.41

Most active user rated: 12952 movies
Most rated movie has: 32831 ratings


In [5]:
global_mean = train["rating"].mean()

# Movie bias
movie_bias = train.groupby("movieId")["rating"].mean() - global_mean

# User bias
user_bias = train.groupby("userId")["rating"].mean() - global_mean

print("Global Mean:", global_mean)


Global Mean: 3.5333946


In [6]:
def bias_predict(df):
    return (
        global_mean
        + df["movieId"].map(movie_bias).fillna(0)
        + df["userId"].map(user_bias).fillna(0)
    )


In [7]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

train_small, val_small = train_test_split(train, test_size=0.2, random_state=42)

val_pred = (
    global_mean
    + val_small["movieId"].map(movie_bias).fillna(0)
    + val_small["userId"].map(user_bias).fillna(0)
)

rmse = np.sqrt(mean_squared_error(val_small["rating"], val_pred))

print("Bias Model RMSE:", rmse)


Bias Model RMSE: 0.8688128284516595


In [8]:
user_ids = train["userId"].unique()
movie_ids = train["movieId"].unique()

user_map = {id_: i for i, id_ in enumerate(user_ids)}
movie_map = {id_: i for i, id_ in enumerate(movie_ids)}

train["user_idx"] = train["userId"].map(user_map)
train["movie_idx"] = train["movieId"].map(movie_map)

n_users = len(user_ids)
n_movies = len(movie_ids)

print("Encoded users:", n_users)
print("Encoded movies:", n_movies)


Encoded users: 162541
Encoded movies: 48213


In [9]:
# Latent factors
k = 50  # you can increase to 80-100 later

# Learning rate & regularization
lr = 0.005
reg = 0.02
n_epochs = 10  # start small to test

# Initialize latent factors
P = np.random.normal(0, 0.1, size=(n_users, k))
Q = np.random.normal(0, 0.1, size=(n_movies, k))

# Initialize biases
b_u = np.zeros(n_users, dtype=np.float32)
b_i = np.zeros(n_movies, dtype=np.float32)

# Global mean
mu = train["rating"].mean()

print("MF initialized with k =", k)


MF initialized with k = 50


In [10]:
# Shuffle data
train_array = train[["user_idx","movie_idx","rating"]].to_numpy()

for epoch in range(n_epochs):
    np.random.shuffle(train_array)
    total_loss = 0
    for u, i, r in train_array:
        u = int(u)
        i = int(i)
        
        pred = mu + b_u[u] + b_i[i] + P[u,:].dot(Q[i,:])
        err = r - pred
        
        # Update biases
        b_u[u] += lr * (err - reg * b_u[u])
        b_i[i] += lr * (err - reg * b_i[i])
        
        # Update latent factors
        P[u,:] += lr * (err * Q[i,:] - reg * P[u,:])
        Q[i,:] += lr * (err * P[u,:] - reg * Q[i,:])
        
        total_loss += err**2
    
    rmse_epoch = np.sqrt(total_loss / len(train_array))
    print(f"Epoch {epoch+1}/{n_epochs} RMSE: {rmse_epoch:.4f}")


Epoch 1/10 RMSE: 0.9288
Epoch 2/10 RMSE: 0.8842
Epoch 3/10 RMSE: 0.8704
Epoch 4/10 RMSE: 0.8621
Epoch 5/10 RMSE: 0.8553
Epoch 6/10 RMSE: 0.8476
Epoch 7/10 RMSE: 0.8384
Epoch 8/10 RMSE: 0.8284
Epoch 9/10 RMSE: 0.8183
Epoch 10/10 RMSE: 0.8080


In [11]:
# --- Ensure test has index mappings---

if "user_idx" not in test.columns:
    test["user_idx"] = test["userId"].map(user_map)

if "movie_idx" not in test.columns:
    test["movie_idx"] = test["movieId"].map(movie_map)


# --- Initialize predictions with global mean ---
preds = np.full(len(test), mu)


# --- Mask: only rows where both user and movie were seen in training ---
mask = (
    test["user_idx"].notna() &
    test["movie_idx"].notna()
)


# --- Safe integer conversion ---
u_idx = test.loc[mask, "user_idx"].astype(int).values
i_idx = test.loc[mask, "movie_idx"].astype(int).values


# --- Matrix Factorization prediction ---
preds[mask] = (
    mu
    + b_u[u_idx]
    + b_i[i_idx]
    + np.sum(P[u_idx] * Q[i_idx], axis=1)
)


# --- Optional: clip to rating range ---
preds = np.clip(preds, 0.5, 5.0)


# --- Assign back ---
test["rating"] = preds


In [12]:

# LOAD MOVIES + PREP GENRES

movies = pd.read_csv('/kaggle/input/movie-recommendation-hackathon-2026/movies.csv')
movies['genres'] = movies['genres'].apply(lambda x: x.split('|'))

# BUILD GENRE BIAS FROM TRAIN

train_genre = train.merge(movies[['movieId','genres']], on='movieId', how='left')
train_genre = train_genre.explode('genres')

genre_mean = train_genre.groupby('genres')['rating'].mean()
genre_bias = genre_mean - mu  # deviation from global mean

# BUILD MOVIE → GENRE MAP

movie_genre_map = movies.set_index('movieId')['genres'].to_dict()


# PREDICTION WITH GENRE-AWARE COLD START

preds = np.full(len(test), mu)

# mask for movies seen in training
seen_mask = (
    test["user_idx"].notna() &
    test["movie_idx"].notna()
)

# Safe integer conversion
u_idx = test.loc[seen_mask, "user_idx"].astype(int).values
i_idx = test.loc[seen_mask, "movie_idx"].astype(int).values

# Standard MF prediction for seen users/movies
preds[seen_mask] = (
    mu
    + b_u[u_idx]
    + b_i[i_idx]
    + np.sum(P[u_idx] * Q[i_idx], axis=1)
)

# GENRE-BASED FALLBACK FOR UNSEEN MOVIES

unseen_mask = ~seen_mask

for idx in test[unseen_mask].index:
    movie_id = test.loc[idx, "movieId"]
    genres = movie_genre_map.get(movie_id, [])

    if len(genres) > 0:
        gb = np.mean([genre_bias.get(g, 0) for g in genres])
        preds[idx] = mu + gb
    else:
        preds[idx] = mu


# CLIP RATINGS TO VALID RANGE

preds = np.clip(preds, 0.5, 5.0)

test["rating"] = preds


In [13]:
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler

# LOAD GENOME DATA

genome_scores = pd.read_csv(
    '/kaggle/input/movie-recommendation-hackathon-2026/genome_scores.csv',
    dtype={'movieId': np.int32, 'tagId': np.int16, 'relevance': np.float32}
)

# Pivot to movieId × tag matrix
movie_tag_matrix = genome_scores.pivot(
    index='movieId',
    columns='tagId',
    values='relevance'
).fillna(0)

del genome_scores


In [14]:
# SCALE + DIMENTIONALITY  REDUCTION
scaler = StandardScaler(with_mean=True, with_std=True)
movie_tag_scaled = scaler.fit_transform(movie_tag_matrix)

n_components = 50   # tune later (50–100)
svd = TruncatedSVD(n_components=n_components, random_state=42)
movie_embeddings = svd.fit_transform(movie_tag_scaled)

# Build embedding lookup
movie_ids_genome = movie_tag_matrix.index.values
movie_embed_map = dict(zip(movie_ids_genome, movie_embeddings))

del movie_tag_matrix, movie_tag_scaled  # free RAM

In [15]:
# ENSURE TEST HAS PROPER INDICES

test["user_idx"] = test["userId"].map(user_map).fillna(-1).astype(np.int32)
test["movie_idx"] = test["movieId"].map(movie_map).fillna(-1).astype(np.int32)

In [16]:

# MATRIX FACTORIZATION PREDICTIONS (VECTORISED)

preds = np.full(len(test), mu, dtype=np.float32)

mf_mask = (test["user_idx"] != -1) & (test["movie_idx"] != -1)

u_idx = test.loc[mf_mask, "user_idx"].values
i_idx = test.loc[mf_mask, "movie_idx"].values

mf_preds = (
    mu
    + b_u[u_idx]
    + b_i[i_idx]
    + np.sum(P[u_idx] * Q[i_idx], axis=1)
)

preds[mf_mask] = mf_preds

In [17]:
# GENOME CONTENT SIGNAL 

content_preds = np.full(len(test), mu, dtype=np.float32)

for idx, movie_id in enumerate(test["movieId"].values):
    vec = movie_embed_map.get(movie_id)
    if vec is not None:
        content_preds[idx] = mu + np.mean(vec)


In [18]:
# HYBRID BLEND

alpha = 0.85   # MF dominates, genome assists

final_preds = alpha * preds + (1 - alpha) * content_preds

# Clip to valid rating range
test["rating"] = np.clip(final_preds, 0.5, 5.0)

# SUBMISSION FILE

test["Id"] = test["userId"].astype(str) + "_" + test["movieId"].astype(str)
submission = test[["Id", "rating"]]

submission.to_csv("submission.csv", index=False)

In [19]:
# Ensure numpy arrays
preds = np.asarray(preds)
content_preds = np.asarray(content_preds)

# Safety check
assert len(preds) == len(content_preds), "Prediction arrays mismatch!"

# HYBRID BLEND (Vectorized)
alpha = 0.85

final_preds = preds * alpha + content_preds * (1 - alpha)

# Clip
final_preds = np.clip(final_preds, 0.5, 5.0)

# Assign
test["rating"] = final_preds

# Build submission
test["Id"] = test["userId"].astype(str) + "_" + test["movieId"].astype(str)
submission = test[["Id", "rating"]]

submission.to_csv("submission.csv", index=False)

print("Submission file created successfully.")


Submission file created successfully.
